In [1]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [2]:
load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

query = '''
SELECT *
FROM hpce.churn_model_dataset
WHERE feature_date <= '2018-08-04'
'''
df = pd.read_sql(query, engine)

df["feature_date"] = pd.to_datetime(df["feature_date"]).dt.date
print(df.shape)
print("Date range:", df["feature_date"].min(), "->", df["feature_date"].max())

(342173, 26)
Date range: 2016-09-04 -> 2018-08-04


In [3]:
FEATURES = [
    "daily_orders",
    "daily_spend",
    "f_30",
    "m_30",
    "f_trend_30",
    "m_trend_30",
    "recency_days"
]
TARGET = "churn_30d"

In [4]:
cutoff = pd.to_datetime("2018-07-01").date()

train = df[df["feature_date"] <= cutoff].copy()
test  = df[df["feature_date"] > cutoff].copy()

print("Train rows:", len(train))
print("Test rows :", len(test))
print("Train date max:", train["feature_date"].max(), "| Test date min:", test["feature_date"].min())

Train rows: 327448
Test rows : 14725
Train date max: 2018-07-01 | Test date min: 2018-07-02


In [5]:
from lightgbm import LGBMClassifier

# Train LightGBM on churn_30d
X_train = train[FEATURES]
y_train = train[TARGET].astype(int)

X_test  = test[FEATURES]
y_test  = test[TARGET].astype(int)

lgbm_model = LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

lgbm_model.fit(X_train, y_train)

test_probs = lgbm_model.predict_proba(X_test)[:, 1]

# quick sanity metrics
from sklearn.metrics import roc_auc_score, average_precision_score
print("LGBM ROC-AUC:", roc_auc_score(y_test, test_probs))
print("LGBM PR-AUC :", average_precision_score(y_test, test_probs))

[LightGBM] [Info] Number of positive: 278426, number of negative: 49022
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003709 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1041
[LightGBM] [Info] Number of data points in the train set: 327448, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
LGBM ROC-AUC: 0.9129400893311181
LGBM PR-AUC : 0.9476543686348815


In [8]:
df_decision = test[["customer_id", "feature_date", TARGET] + FEATURES].copy()
df_decision["churn_prob"] = test_probs

df_decision[["customer_id", "feature_date", "churn_prob", TARGET]].head()
# ============================
# (NEW) Join Synthetic Event Features (Decision Engine Only)
# ============================

query_events = """
SELECT
  customer_id,
  feature_date,
  views_7,
  views_30,
  searches_30,
  carts_30,
  events_30,
  cart_to_view_rate_30,
  event_recency_days
FROM hpce.synthetic_event_features_daily
WHERE feature_date BETWEEN '2018-07-02' AND '2018-08-04';
"""
df_events = pd.read_sql(query_events, engine)

# Ensure same dtype as df_decision
df_events["feature_date"] = pd.to_datetime(df_events["feature_date"]).dt.date

# Merge (left join so you never lose decision rows)
df_decision = df_decision.merge(
    df_events,
    on=["customer_id", "feature_date"],
    how="left"
)

# Fill missing (if a customer has no synthetic events that day)
event_cols = ["views_7","views_30","searches_30","carts_30","events_30","cart_to_view_rate_30","event_recency_days"]
for c in event_cols:
    df_decision[c] = df_decision[c].fillna(0 if c != "event_recency_days" else 999)

df_decision.head()


,customer_id,feature_date,churn_30d,daily_orders,daily_spend,f_30,m_30,f_trend_30,m_trend_30,recency_days,churn_prob,views_7,views_30,searches_30,carts_30,events_30,cart_to_view_rate_30,event_recency_days
0,000fbf0473c10fc1ab6f8d2d286ce20c,2018-07-26,1,1,362.36,1.0,362.36,1.0,362.36,0,0.965014,4.0,4.0,4.0,0.0,8.0,0.00,0
1,0015752e079902b12cd00b9b7596276b,2018-08-02,1,1,74.82,1.0,74.82,1.0,74.82,0,0.973430,3.0,3.0,4.0,0.0,7.0,0.00,0
2,00172711b30d52eea8b313a7f2cced02,2018-07-28,1,1,122.07,1.0,122.07,1.0,122.07,0,0.949031,4.0,4.0,1.0,2.0,7.0,0.50,0
3,00196c4c9a3af7dd2ad10eade69c926f,2018-07-28,1,1,37.58,1.0,37.58,1.0,37.58,0,0.960135,5.0,5.0,1.0,0.0,6.0,0.00,0
4,001a34eb30ecb8e3aacb07c475ca4dd1,2018-07-11,1,1,161.71,1.0,161.71,1.0,161.71,0,0.970791,4.0,4.0,1.0,1.0,6.0,0.25,0


In [9]:
# Dynamic band thresholds (quantile-based) so bands remain stable across models
# High Risk = top 30% probabilities, Medium = next 30%, Low = bottom 40%
high_th = df_decision["churn_prob"].quantile(0.70)
med_th  = df_decision["churn_prob"].quantile(0.40)

print("Risk thresholds:", {"high_th": float(high_th), "med_th": float(med_th)})

def assign_risk(p):
    if p >= high_th:
        return "High Risk"
    elif p >= med_th:
        return "Medium Risk"
    else:
        return "Low Risk"

df_decision["risk_band"] = df_decision["churn_prob"].apply(assign_risk)

Risk thresholds: {'high_th': 0.9601701665934906, 'med_th': 0.45772975959864776}


In [10]:
band_churn = df_decision.groupby("risk_band")[TARGET].mean().sort_values(ascending=False)
band_counts = df_decision["risk_band"].value_counts()

print("Counts:\n", band_counts)
print("\nChurn rate by band:\n", band_churn)

Counts:
 risk_band
Low Risk       5775
Medium Risk    4514
High Risk      4436
Name: count, dtype: int64

Churn rate by band:
 risk_band
High Risk      0.987827
Medium Risk    0.684759
Low Risk       0.194113
Name: churn_30d, dtype: float64


In [13]:
def assign_action(row):
    band = row["risk_band"]
    low_engagement = (row["events_30"] == 0) or (row["event_recency_days"] > 14)

    if band == "High Risk":
        # if high risk and low engagement -> stronger retention action
        return "Retention Offer (Urgent)" if low_engagement else "Retention Offer"

    elif band == "Medium Risk":
        # medium risk but low engagement -> nurture
        return "Engagement Campaign (Nudge)" if low_engagement else "Engagement Campaign"

    else:
        # low risk but suddenly no engagement -> monitor
        return "Monitor" if low_engagement else "No Action"

df_decision["action"] = df_decision.apply(assign_action, axis=1)
df_decision[["customer_id","feature_date","churn_prob","risk_band","action",TARGET]].head(15)

,customer_id,feature_date,churn_prob,risk_band,action,churn_30d
0,000fbf0473c10fc1ab6f8d2d286ce20c,2018-07-26,0.965014,High Risk,Retention Offer,1
1,0015752e079902b12cd00b9b7596276b,2018-08-02,0.973430,High Risk,Retention Offer,1
2,00172711b30d52eea8b313a7f2cced02,2018-07-28,0.949031,Medium Risk,Engagement Campaign,1
3,00196c4c9a3af7dd2ad10eade69c926f,2018-07-28,0.960135,Medium Risk,Engagement Campaign,1
4,001a34eb30ecb8e3aacb07c475ca4dd1,2018-07-11,0.970791,High Risk,Retention Offer,1
5,0025795df7a7d077c4c90162fa820085,2018-07-24,0.959094,Medium Risk,Engagement Campaign,1
6,003162bf0351f47a1a247992b8b9b42d,2018-07-14,0.992097,High Risk,Retention Offer,1
7,0032ca69771a7758ce643258bb9e4ac7,2018-07-16,0.969551,High Risk,Retention Offer,1
8,00344274804f3b8003de1b0562ae01df,2018-07-09,0.963873,High Risk,Retention Offer,1
9,003a5571a07dcf09bf117d13d2980ba3,2018-07-25,0.950097,Medium Risk,Engagement Campaign,1


In [15]:
band_priority = {"High Risk": 2, "Medium Risk": 1, "Low Risk": 0}

df_decision["priority_band"] = df_decision["risk_band"].map(band_priority)
df_decision["priority_score"] = df_decision["priority_band"] * 10 + df_decision["churn_prob"]

df_decision.sort_values(["priority_band", "churn_prob"], ascending=[False, False])[
    ["customer_id","feature_date","churn_prob","risk_band","action","priority_score",TARGET]
]

,customer_id,feature_date,churn_prob,risk_band,action,priority_score,churn_30d
5755,62b8761192c86f4ead06f4a433096a2b,2018-07-04,0.993468,High Risk,Retention Offer,20.993468,1
2403,2be7d578854588f75b1922cf7627e1be,2018-07-22,0.992746,High Risk,Retention Offer,20.992746,1
2638,301fdbdb88825096a683fcfacac44718,2018-07-14,0.992746,High Risk,Retention Offer,20.992746,1
4611,525cda9909aa001ebed396f6e55eae01,2018-07-13,0.992746,High Risk,Retention Offer,20.992746,0
6238,6d887fdae7b23b09c8108f16900dab42,2018-07-20,0.992746,High Risk,Retention Offer,20.992746,1
...,...,...,...,...,...,...,...
8169,8d50f5eadf50201ccdcedfb9e2ac8455,2018-08-02,0.180385,Low Risk,No Action,0.180385,0
8168,8d50f5eadf50201ccdcedfb9e2ac8455,2018-08-01,0.176737,Low Risk,No Action,0.176737,0
8167,8d50f5eadf50201ccdcedfb9e2ac8455,2018-07-31,0.175995,Low Risk,No Action,0.175995,0
8166,8d50f5eadf50201ccdcedfb9e2ac8455,2018-07-30,0.175379,Low Risk,No Action,0.175379,0


In [28]:
# ============================
# Persist decision engine output + intervention logs
# ============================

import pandas as pd

# Ensure feature_date is date (not datetime)
df_decision["feature_date"] = pd.to_datetime(df_decision["feature_date"]).dt.date

# 1) Save full decision output
cols_out = [
    "customer_id","feature_date",
    "churn_prob","risk_band","action","priority_score",
    "views_7","views_30","searches_30","carts_30","events_30",
    "cart_to_view_rate_30","event_recency_days"
]
df_out = df_decision[cols_out].copy()

# Write to hpce.decision_engine_output_daily (replace for your scoring window)
df_out.to_sql(
    "decision_engine_output_daily",
    engine,
    schema="hpce",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000
)

print("Inserted decision_engine_output_daily rows:", len(df_out))


# 2) Save intervention logs (feedback loop "action record")
# For now, intervention_date = feature_date (system decision day)
df_logs = df_out[["customer_id","feature_date","action","churn_prob","risk_band","priority_score","events_30","event_recency_days"]].copy()
df_logs = df_logs.rename(columns={"feature_date":"intervention_date"})
df_logs["feature_date"] = df_logs["intervention_date"]

df_logs.to_sql(
    "intervention_logs",
    engine,
    schema="hpce",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000
)

print("Inserted intervention_logs rows:", len(df_logs))


Inserted decision_engine_output_daily rows: 14725
Inserted intervention_logs rows: 14725
